In [1]:
!git clone https://github.com/ahmedalashii/Yolov11-Ingredients-Detection.git

Cloning into 'Yolov11-Ingredients-Detection'...
remote: Enumerating objects: 4806, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4806 (delta 2), reused 6 (delta 2), pack-reused 4800 (from 1)
Receiving objects: 100% (4806/4806), 338.30 MiB | 23.81 MiB/s, done.
Resolving deltas: 100% (45/45), done.
Updating files: 100% (8557/8557), done.


In [2]:
%cd Yolov11-Ingredients-Detection

/content/Yolov11-Ingredients-Detection


In [3]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.2 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
ROOT_DIR = os.getcwd()

# Dataset YAML path
SINGLE_INGREDIENT_DATA_YAML_PATH = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'data.yaml')

# Pretrained model path
ORIGINAL_YOLO_MODEL_PATH = os.path.join(ROOT_DIR, 'models', 'original_yolo11s.pt')

# Output dir for saving trained models
OUTPUT_DIR = os.path.join(ROOT_DIR, 'models')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
import yaml

with open(SINGLE_INGREDIENT_DATA_YAML_PATH, 'r') as file:
    class_names = yaml.safe_load(file)['names']
print("Class names loaded from YAML file:", class_names)
print(f"Number of classes: {len(class_names)}")

Class names loaded from YAML file: ['banana', 'bell pepper', 'bulgur', 'cabbage', 'carrot', 'cheese', 'chicken', 'chickpeas', 'cucumber', 'egg', 'eggplant', 'farfalle pasta', 'fusilli pasta', 'garlic', 'green pepper', 'ground meat', 'ketchup', 'lemon', 'margarine', 'mayonnaise', 'milk', 'mushroom', 'olive', 'olive oil', 'onion', 'parsley', 'potato', 'red lentils', 'rice', 'sausage', 'spaghetti', 'tomato', 'tomato paste', 'turkish dumplings', 'yogurt', 'zucchini']
Number of classes: 36


In [7]:
import yaml
import os

# Load the data.yaml file
with open(SINGLE_INGREDIENT_DATA_YAML_PATH, 'r') as file:
    data = yaml.safe_load(file)

class_names = data['names']
num_classes = len(class_names)

# Function to count instances per class in a given directory
def count_classes(label_dir, class_names):
    class_counts = {name: 0 for name in class_names}
    if not os.path.exists(label_dir):
        print(f"Warning: Directory not found: {label_dir}")
        return class_counts

    for label_file in os.listdir(label_dir):
        if label_file.endswith('.txt'):
            with open(os.path.join(label_dir, label_file), 'r') as f:
                for line in f:
                    try:
                        class_id = int(line.split()[0])
                        if 0 <= class_id < len(class_names):
                            class_name = class_names[class_id]
                            class_counts[class_name] += 1
                        else:
                            print(f"Warning: Invalid class ID {class_id} in file {label_file}")
                    except (ValueError, IndexError):
                        print(f"Warning: Could not parse line in file {label_file}: {line.strip()}")
    return class_counts

# Define paths to label directories for each split
train_label_dir = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'train', 'labels')
val_label_dir = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'valid', 'labels')
test_label_dir = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'test', 'labels')


# Get the path to the training, validation, and test images
train_img_dir = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'train', 'images')
val_img_dir = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'valid', 'images')
test_img_dir = os.path.join(ROOT_DIR, 'datasets', 'yolo_single_ingredient_dataset', 'test', 'images')


# Count instances for each split
train_class_counts = count_classes(train_label_dir, class_names)
val_class_counts = count_classes(val_label_dir, class_names)
test_class_counts = count_classes(test_label_dir, class_names)

# Calculate overall counts
overall_class_counts = {name: train_class_counts[name] + val_class_counts[name] + test_class_counts[name] for name in class_names}


# Print results
print("Class Distribution across Splits:")
print("\nTraining Set:")
print("Number of images:", len(os.listdir(train_img_dir)))
print("Number of instances per class:")
for class_name, count in train_class_counts.items():
    print(f"{class_name:<20}: {count}")

print("\nValidation Set:")
print("Number of images:", len(os.listdir(val_img_dir)))
print("Number of instances per class:")
for class_name, count in val_class_counts.items():
    print(f"{class_name:<20}: {count}")

print("\nTest Set:")
print("Number of images:", len(os.listdir(test_img_dir)))
print("Number of instances per class:")
for class_name, count in test_class_counts.items():
    print(f"{class_name:<20}: {count}")


print("\nOverall Dataset:")
print("Number of images:", len(os.listdir(train_img_dir)) + len(os.listdir(val_img_dir)) + len(os.listdir(test_img_dir)))
print("Number of instances per class:")
for class_name, count in overall_class_counts.items():
    print(f"{class_name:<20}: {count}")

Class Distribution across Splits:

Training Set:
Number of images: 1425
Number of instances per class:
banana              : 87
bell pepper         : 151
bulgur              : 15
cabbage             : 11
carrot              : 16
cheese              : 63
chicken             : 20
chickpeas           : 32
cucumber            : 118
egg                 : 254
eggplant            : 81
farfalle pasta      : 18
fusilli pasta       : 18
garlic              : 126
green pepper        : 122
ground meat         : 18
ketchup             : 20
lemon               : 351
margarine           : 24
mayonnaise          : 21
milk                : 99
mushroom            : 261
olive               : 189
olive oil           : 69
onion               : 264
parsley             : 19
potato              : 360
red lentils         : 16
rice                : 76
sausage             : 16
spaghetti           : 20
tomato              : 325
tomato paste        : 80
turkish dumplings   : 13
yogurt              : 28
zucchini   

In [8]:
import pandas as pd

# Create a DataFrame from the class counts
df_class_counts = pd.DataFrame({
    'Class Name': class_names,
    'Training Set': train_class_counts.values(),
    'Validation Set': val_class_counts.values(),
    'Test Set': test_class_counts.values(),
    'Overall Dataset': overall_class_counts.values()
})

# Display the DataFrame
print("Class Distribution across Splits:")
display(df_class_counts)

Class Distribution across Splits:


,Class Name,Training Set,Validation Set,Test Set,Overall Dataset
0,banana,87,12,11,110
1,bell pepper,151,26,35,212
2,bulgur,15,1,2,18
3,cabbage,11,2,3,16
4,carrot,16,5,2,23
5,cheese,63,21,7,91
6,chicken,20,13,1,34
7,chickpeas,32,4,4,40
8,cucumber,118,26,17,161
9,egg,254,37,52,343


In [9]:
import os
import cv2
import random
import numpy as np
from glob import glob

# -------------------------------
# CONFIG
# -------------------------------
single_root = "/content/Yolov11-Ingredients-Detection/datasets/all_single_ingredients"  # has images/ and labels/
images_dir = os.path.join(single_root, "images")
labels_dir = os.path.join(single_root, "labels")

output_root = "/content/multi_ingredient_dataset"
os.makedirs(os.path.join(output_root, "valid/images"), exist_ok=True)
os.makedirs(os.path.join(output_root, "valid/labels"), exist_ok=True)

CANVAS_SIZE = (640, 640)  # (width, height)

# -------------------------------
# LOAD SOURCE DATA
# -------------------------------
all_images = glob(os.path.join(images_dir, "*.jpg")) + glob(os.path.join(images_dir, "*.png"))

# Map each image to its label
image_label_map = {}
for img_path in all_images:
    base = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(labels_dir, base + ".txt")
    if os.path.exists(label_path):
        image_label_map[img_path] = label_path

if not image_label_map:
    raise RuntimeError("No image-label pairs found. Check your folder structure and extensions.")

# -------------------------------
# GENERATION FUNCTION
# -------------------------------
def create_multi_image(num_ingredients=3, n_samples=100):
    w, h = CANVAS_SIZE
    for idx in range(n_samples):
        canvas = 255 * np.ones((h, w, 3), dtype=np.uint8)
        label_lines = []

        available_imgs = list(image_label_map.keys())
        k = min(num_ingredients, len(available_imgs))
        chosen_imgs = random.sample(available_imgs, k)

        for img_path in chosen_imgs:
            label_path = image_label_map[img_path]
            img = cv2.imread(img_path)
            if img is None:
                continue

            orig_h, orig_w = img.shape[:2]

            # Random resize
            scale = random.uniform(0.4, 0.8)
            new_w = int(orig_w * scale)
            new_h = int(orig_h * scale)
            if new_w < 2 or new_h < 2:
                continue
            img_resized = cv2.resize(img, (new_w, new_h))

            # Random position
            max_x = max(1, w - new_w)
            max_y = max(1, h - new_h)
            x_offset = random.randint(0, max_x)
            y_offset = random.randint(0, max_y)

            # Paste ingredient
            canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = img_resized

            # Parse and transform labels
            with open(label_path, "r") as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    try:
                        cls, x, y, bw, bh = map(float, parts[:5])
                    except ValueError:
                        continue
                    cls = int(cls)

                    # Convert normalized to absolute (original image)
                    abs_x = x * orig_w
                    abs_y = y * orig_h
                    abs_bw = bw * orig_w
                    abs_bh = bh * orig_h

                    # Scale to resized image
                    abs_x *= scale
                    abs_y *= scale
                    abs_bw *= scale
                    abs_bh *= scale

                    # Shift to canvas and normalize again
                    new_x = (x_offset + abs_x) / w
                    new_y = (y_offset + abs_y) / h
                    new_bw = abs_bw / w
                    new_bh = abs_bh / h

                    label_lines.append(f"{cls} {new_x:.6f} {new_y:.6f} {new_bw:.6f} {new_bh:.6f}")

        # Save outputs
        out_img_path = os.path.join(output_root, "valid/images", f"multi_{idx:04d}.jpg")
        out_lbl_path = os.path.join(output_root, "valid/labels", f"multi_{idx:04d}.txt")
        cv2.imwrite(out_img_path, canvas)
        with open(out_lbl_path, "w") as f:
            f.write("\n".join(label_lines))


# -------------------------------
# RUN
# -------------------------------
create_multi_image(num_ingredients=2, n_samples=1500)
print("Synthetic dataset created at:", output_root)

Synthetic dataset created at: /content/multi_ingredient_dataset


In [ ]:
# Original YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_PATH = os.path.join(ROOT_DIR, 'models', 'original_yolo11s.pt')

# Load the original model
original_model = YOLO(YOLO_MODEL_PATH)

# Evaluate the model
metrics = original_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_original_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = original_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx < len(class_names) else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 698.3±233.5 MB/s, size: 58.9 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 1.7Kit/s 0.2s
val: New cache created: /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.7it/s 6.7s
                   all        286        728    0.00751     0.0849    0.00807    0.00488
                person          8         12    0.00156     0.0833    0.00088   0.000352
               bicycle          6         26          0          0          0          0
                   car          1          1   

In [11]:
MULTIPLE_INGREDIENTS_DATA_YAML_PATH = os.path.join('/content', 'multi_ingredient_dataset', 'data.yaml')
# Print if it exists or not
if os.path.exists(MULTIPLE_INGREDIENTS_DATA_YAML_PATH):
    print(f"The file '{MULTIPLE_INGREDIENTS_DATA_YAML_PATH}' exists.")

The file '/content/multi_ingredient_dataset/data.yaml' exists.


In [ ]:
# Original YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_PATH = os.path.join(ROOT_DIR, 'models', 'original_yolo11s.pt')

# Load the original model
original_model = YOLO(YOLO_MODEL_PATH)

# Evaluate the model
metrics = original_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_original_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = original_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx < len(class_names) else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2063.7±939.4 MB/s, size: 78.2 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 2.0Kit/s 0.8s
val: New cache created: /content/multi_ingredient_dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.3it/s 21.7s
                   all       1500       7331    0.00387     0.0248    0.00281    0.00142
                person        132        206    0.00188     0.0485    0.00103   0.000527
               bicycle         99        369    0.00177    0.00271   0.000897   8.97e-05
                   car         22         22          0          0          0          0
            motorcycle         11         2

In [15]:
import shutil
import os

# Define the path to the models directory
models_dir = os.path.join(ROOT_DIR, 'models')

# Get a list of all items in the models directory
items_in_models = os.listdir(models_dir)

# Iterate over the items and zip directories
for item in items_in_models:
    item_path = os.path.join(models_dir, item)
    if os.path.isdir(item_path):
        zip_output_path = os.path.join(models_dir, f'{item}.zip')
        shutil.make_archive(base_name=zip_output_path.replace('.zip', ''), format='zip', root_dir=item_path)
        print(f"Zipped folder: {item_path} to {zip_output_path}")

print("All folders inside the 'models' directory have been zipped.")

Zipped folder: /content/Yolov11-Ingredients-Detection/models/yolo11s_60_epochs to /content/Yolov11-Ingredients-Detection/models/yolo11s_60_epochs.zip
Zipped folder: /content/Yolov11-Ingredients-Detection/models/yolo11s_30_epochs to /content/Yolov11-Ingredients-Detection/models/yolo11s_30_epochs.zip
Zipped folder: /content/Yolov11-Ingredients-Detection/models/A3_strong_aug_yolo11s_100_epochs to /content/Yolov11-Ingredients-Detection/models/A3_strong_aug_yolo11s_100_epochs.zip
Zipped folder: /content/Yolov11-Ingredients-Detection/models/yolo11s_100_epochs to /content/Yolov11-Ingredients-Detection/models/yolo11s_100_epochs.zip
All folders inside the 'models' directory have been zipped.


# Augmentation Policies:

	•	A0 (None): no augmentation (only resize + normalize).
	•	A1 (Basic): horizontal flip (50%), random crop ±10%, random brightness/contrast.
	•	A2 (Extended): A1 + rotation ±15°, scale (0.8–1.2), color jitter (hue/saturation).
	•	A3 (Strong / Ultralytics default): Ultralytics advanced augmentations (mosaic, mixup, HSV jitter, random perspective, copy-paste if available).

In [ ]:
# A0 — None (no augmentation)
model_30 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_30.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=30, imgsz=640, batch=32, project=OUTPUT_DIR, name='A0_no_aug_yolo11s_30_epochs', augment=False)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=A0_no_aug_yolo11s_30_ep

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d726085dd00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A1 — Basic (flip, crop, brightness/contrast)
model_30 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_30.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=30, imgsz=640, batch=32, project=OUTPUT_DIR, name='A1_basic_aug_yolo11s_30_epochs', augment=True, fliplr=0.5, flipud=0.0, scale=0.0, degrees=0.0, translate=0.1, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, shear=0.0, mixup=0.0, mosaic=0.0)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=A1_basic_aug_yolo11s_30_ep

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d72f51ed070>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A2 (Extended (rotation, scale, color jitter))
model_30 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_30.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=30, imgsz=640, batch=32, project=OUTPUT_DIR, name='A2_extended_aug_yolo11s_30_epochs', augment=True, fliplr=0.5, flipud=0.0, degrees=15.0, scale=0.2, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, shear=0.0, mixup=0.0, mosaic=0.0)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=A2_extended_aug_yolo11s

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7adf2713f740>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A3 (Strong / Ultralytics default)
model_30 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_30.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=30, imgsz=640, batch=32, project=OUTPUT_DIR, name='A3_strong_aug_yolo11s_30_epochs', augment=True,   mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    degrees=10.0,
    scale=0.9,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    fliplr=0.5)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=A3_strong_aug_yolo11s_3

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ae05db9e510>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A0 — None (no augmentation)
model_60 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_60.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=60, imgsz=640, batch=32, project=OUTPUT_DIR, name='A0_no_aug_yolo11s_60_epochs', augment=False)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=A0_no_aug_yolo11s_60_ep

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d7260871730>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A1 — Basic (flip, crop, brightness/contrast)
model_60 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_60.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=60, imgsz=640, batch=32, project=OUTPUT_DIR, name='A1_basic_aug_yolo11s_60_epochs', augment=True, fliplr=0.5, flipud=0.0, scale=0.0, degrees=0.0, translate=0.1, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, shear=0.0, mixup=0.0, mosaic=0.0)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=A1_basic_aug_yolo11s_60_ep

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bccf6432d80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A2 (Extended (rotation, scale, color jitter))
model_60 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_60.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=60, imgsz=640, batch=32, project=OUTPUT_DIR, name='A2_extended_aug_yolo11s_60_epochs', augment=True, fliplr=0.5, flipud=0.0, degrees=15.0, scale=0.2, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, shear=0.0, mixup=0.0, mosaic=0.0)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=A2_extended_aug_yolo11s

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x785f564f8860>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A3 (Strong / Ultralytics default)
model_60 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_60.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=60, imgsz=640, batch=32, project=OUTPUT_DIR, name='A3_strong_aug_yolo11s_60_epochs', augment=True,   mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    degrees=10.0,
    scale=0.9,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    fliplr=0.5)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=A3_strong_aug_yolo11s_6

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x785f444c76e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A0 — None (no augmentation)
model_100 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_100.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=100, imgsz=640, batch=32, project=OUTPUT_DIR, name='A0_no_aug_yolo11s_100_epochs', augment=False)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=A0_no_aug_yolo11s_100_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x792ac83f9e80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A1 — Basic (flip, crop, brightness/contrast)
model_100 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_100.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=100, imgsz=640, batch=32, project=OUTPUT_DIR, name='A1_basic_aug_yolo11s_100_epochs', augment=True, fliplr=0.5, flipud=0.0, scale=0.0, degrees=0.0, translate=0.1, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, shear=0.0, mixup=0.0, mosaic=0.0)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=A1_basic_aug_yolo11s_100_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x792aa22ae060>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A2 (Extended (rotation, scale, color jitter))
model_100 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_100.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=100, imgsz=640, batch=32, project=OUTPUT_DIR, name='A2_extended_aug_yolo11s_100_epochs', augment=True, fliplr=0.5, flipud=0.0, degrees=15.0, scale=0.2, translate=0.1, hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, shear=0.0, mixup=0.0, mosaic=0.0)

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.7, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=A2_extended_aug_yolo11

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x792cc2df12b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [12]:
# A3 (Strong / Ultralytics default)
model_100 = YOLO(ORIGINAL_YOLO_MODEL_PATH)
model_100.train(data=SINGLE_INGREDIENT_DATA_YAML_PATH, epochs=100, imgsz=640, batch=32, project=OUTPUT_DIR, name='A3_strong_aug_yolo11s_100_epochs', augment=True,   mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    degrees=10.0,
    scale=0.9,
    shear=2.0,
    perspective=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    fliplr=0.5)

Ultralytics 8.3.211 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=/content/Yolov11-Ingredients-Detection/models/original_yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=A3_strong_aug_yolo11s_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7888befa5730>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038, 

In [ ]:
# A0 No Aug - 30 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A0_no_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A0_no_aug_yolo11s_30_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1856.5±914.0 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 561.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.0it/s 6.1s
                   all        286        728      0.864      0.801      0.859      0.682
                banana          8         12      0.787       0.75      0.747       0.49
           bell pepper          6         26      0.848      0.962      0.946      0.743
                bulgur          1          1      0.941          1      0.995      0.995
               cabbage          1          2      0.929          1     

In [ ]:
# A1 Basic Aug - 30 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A1_basic_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A1_basic_aug_yolo11s_30_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1902.2±572.6 MB/s, size: 57.2 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 551.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.7s
                   all        286        728      0.814      0.781      0.843      0.639
                banana          8         12       0.65      0.667      0.785      0.508
           bell pepper          6         26      0.817      0.862      0.932      0.744
                bulgur          1          1      0.954          1      0.995      0.597
               cabbage          1          2      0.855          1     

In [ ]:
# A2 Extended Aug - 30 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A2_extended_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A2_extended_aug_yolo11s_30_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1415.7±585.3 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 538.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.7s
                   all        286        728      0.837      0.759       0.82      0.614
                banana          8         12      0.744      0.833      0.801      0.456
           bell pepper          6         26      0.774      0.922      0.913      0.692
                bulgur          1          1          1          0      0.249     0.0849
               cabbage          1          2      0.828          1     

In [ ]:
# A3 Strong Aug - 30 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A3_strong_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A3_strong_aug_yolo11s_30_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1672.7±548.5 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 579.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.1it/s 5.8s
                   all        286        728      0.812      0.775      0.871      0.635
                banana          8         12      0.846        0.5      0.788      0.423
           bell pepper          6         26      0.715      0.962      0.939       0.68
                bulgur          1          1          1          0      0.995      0.597
               cabbage          1          2      0.796          1     

In [ ]:
# A0 No Aug - 30 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A0_no_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A0_no_aug_yolo11s_30_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 27.8±9.5 MB/s, size: 88.5 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 472.5it/s 3.2s
val: New cache created: /content/multi_ingredient_dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 3.8it/s 24.9s
                   all       1500       7480      0.866      0.555      0.625      0.493
                banana        119        179      0.727      0.469      0.553      0.433
           bell pepper         89        336      0.823      0.593      0.644      0.519
                bulgur         16         19      0.847      0.526      0.593      0.475
               cabbage         14         28  

In [ ]:
# A1 Basic Aug - 30 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A1_basic_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A1_basic_aug_yolo11s_30_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2318.3±588.6 MB/s, size: 107.0 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 2.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.3it/s 21.9s
                   all       1500       7480      0.766      0.434      0.511      0.376
                banana        119        179      0.692      0.425      0.513      0.354
           bell pepper         89        336      0.902      0.356      0.483      0.359
                bulgur         16         19      0.518      0.368      0.403      0.302
               cabbage         14         28      0.773      0.364      0.425      0.367
                carrot  

In [ ]:
# A2 Extended Aug - 30 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A2_extended_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A2_extended_aug_yolo11s_30_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 412.3±885.7 MB/s, size: 94.7 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 540.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.2it/s 22.3s
                   all       1500       7531      0.748       0.44      0.498      0.355
                banana         99        165      0.594       0.43      0.487      0.303
           bell pepper         97        350      0.813      0.373      0.489      0.343
                bulgur         36         49      0.833      0.367      0.456      0.312
               cabbage         10         21      0.841      0.286       0.41      0.291
                carrot  

In [ ]:
# A3 Strong Aug - 30 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_30_PATH = os.path.join(ROOT_DIR, 'models', 'A3_strong_aug_yolo11s_30_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_30_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A3_strong_aug_yolo11s_30_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2078.1±1211.8 MB/s, size: 94.7 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 3.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.3it/s 21.8s
                   all       1500       7531      0.805      0.543      0.578      0.406
                banana         99        165      0.755      0.461       0.55       0.34
           bell pepper         97        350      0.648      0.577      0.573      0.405
                bulgur         36         49      0.872       0.49      0.571      0.456
               cabbage         10         21      0.932      0.571      0.591       0.49
                carrot  

In [ ]:
# A0 No Aug - 60 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A0_no_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A0_no_aug_yolo11s_60_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1308.9±436.3 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 553.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.3it/s 8.0s
                   all        286        728       0.83      0.791      0.869      0.678
                banana          8         12      0.666       0.75      0.776      0.533
           bell pepper          6         26      0.826      0.885       0.91      0.687
                bulgur          1          1          1          0      0.995      0.895
               cabbage          1          2      0.808          1     

In [ ]:
# A1 Basic Aug - 60 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A1_basic_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A1_basic_aug_yolo11s_60_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1313.0±320.2 MB/s, size: 57.2 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 360.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        286        728      0.868      0.728      0.832      0.651
                banana          8         12      0.815       0.75      0.811      0.552
           bell pepper          6         26      0.784      0.885      0.901      0.704
                bulgur          1          1          1          0      0.995      0.697
               cabbage          1          2       0.92          1     

In [ ]:
# A2 Extended Aug - 60 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A2_extended_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A2_extended_aug_yolo11s_60_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1103.3±416.9 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 301.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.1it/s 5.7s
                   all        286        728      0.805      0.763       0.82      0.636
                banana          8         12      0.712      0.619      0.684      0.431
           bell pepper          6         26        0.7      0.846       0.87      0.674
                bulgur          1          1          1          0          0          0
               cabbage          1          2      0.841          1     

In [ ]:
# A3 Strong Aug - 60 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A3_strong_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A3_strong_aug_yolo11s_60_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1640.0±529.0 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 555.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.4it/s 5.3s
                   all        286        728      0.837      0.816       0.88      0.662
                banana          8         12      0.683      0.719      0.764      0.448
           bell pepper          6         26      0.791      0.871      0.921      0.699
                bulgur          1          1      0.804          1      0.995      0.398
               cabbage          1          2      0.856          1     

In [ ]:
# A0 No Aug - 60 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A0_no_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A0_no_aug_yolo11s_60_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 32.9±10.0 MB/s, size: 104.3 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 2.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 3.5it/s 26.8s
                   all       1500       7480      0.871      0.577      0.625        0.5
                banana        119        179      0.723      0.553       0.56      0.449
           bell pepper         89        336      0.852      0.607      0.651      0.534
                bulgur         16         19      0.843      0.474       0.57      0.425
               cabbage         14         28      0.891        0.5      0.562      0.494
                carrot     

In [ ]:
# A1 Basic Aug - 60 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A1_basic_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A1_basic_aug_yolo11s_60_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 32.3±8.9 MB/s, size: 88.5 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 2.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 3.9it/s 23.9s
                   all       1500       7480      0.775      0.408       0.51      0.379
                banana        119        179      0.814      0.416      0.561      0.397
           bell pepper         89        336      0.884      0.366      0.511      0.387
                bulgur         16         19      0.584      0.211      0.353      0.241
               cabbage         14         28      0.859      0.321       0.42      0.373
                carrot       

In [ ]:
# A2 Extended Aug - 60 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A2_extended_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A2_extended_aug_yolo11s_60_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 33.9±9.3 MB/s, size: 104.3 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 407.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 3.7it/s 25.5s
                   all       1500       7480      0.799      0.438      0.535      0.398
                banana        119        179       0.66      0.467      0.512      0.359
           bell pepper         89        336      0.815      0.407      0.545      0.422
                bulgur         16         19          1      0.342      0.572      0.414
               cabbage         14         28          1      0.368      0.588      0.505
                carrot    

In [ ]:
# A3 Strong Aug - 60 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_60_PATH = os.path.join(ROOT_DIR, 'models', 'A3_strong_aug_yolo11s_60_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_60_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A2_strong_aug_yolo11s_60_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 405.0±503.3 MB/s, size: 104.3 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 2.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.3it/s 21.8s
                   all       1500       7480       0.86      0.577      0.638       0.48
                banana        119        179      0.792      0.492       0.57      0.425
           bell pepper         89        336      0.763      0.595      0.633      0.486
                bulgur         16         19      0.785      0.579      0.608      0.459
               cabbage         14         28      0.837      0.536      0.636      0.477
                carrot   

In [ ]:
# A0 No Aug - 100 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A0_no_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A0_no_aug_yolo11s_100_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1390.2±490.8 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 579.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.9it/s 6.2s
                   all        286        728      0.845      0.811      0.869      0.688
                banana          8         12      0.856       0.75      0.795      0.557
           bell pepper          6         26      0.844      0.846      0.911      0.711
                bulgur          1          1      0.328          1      0.995      0.995
               cabbage          1          2      0.888          1     

In [ ]:
# A1 Basic Aug - 100 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A1_basic_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A1_basic_aug_yolo11s_100_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1637.2±653.7 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 359.2Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.1it/s 5.8s
                   all        286        728      0.857       0.78      0.843      0.666
                banana          8         12      0.626      0.833      0.747      0.532
           bell pepper          6         26      0.797      0.757       0.87      0.688
                bulgur          1          1      0.697          1      0.995      0.597
               cabbage          1          2          1          1     

In [ ]:
# A2 Extended Aug - 100 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A2_extended_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A2_extended_aug_yolo11s_100_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.2 ms, read: 1148.1±359.1 MB/s, size: 63.6 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 444.8Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.0it/s 6.0s
                   all        286        728       0.77      0.841       0.86      0.641
                banana          8         12      0.605       0.75      0.801      0.551
           bell pepper          6         26      0.652      0.846      0.823      0.596
                bulgur          1          1      0.783          1      0.995      0.464
               cabbage          1          2      0.731          1     

In [13]:
# A3 Strong Aug - 100 epochs Fine-tuned YOLO Evaluation on single-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A3_strong_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=SINGLE_INGREDIENT_DATA_YAML_PATH, name='val_A3_strong_aug_yolo11s_100_epochs_single', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping

# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.211 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1113.0±496.8 MB/s, size: 56.7 KB)
val: Scanning /content/Yolov11-Ingredients-Detection/datasets/yolo_single_ingredient_dataset/valid/labels.cache... 286 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 286/286 348.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.6it/s 6.9s
                   all        286        728      0.806      0.806      0.875      0.675
                banana          8         12      0.885      0.646      0.818      0.515
           bell pepper          6         26      0.727      0.885      0.934      0.721
                bulgur          1          1      0.801          1      0.995      0.895
               cabbage          1          2       0.84          1     

In [ ]:
# A0 No Aug - 100 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A0_no_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A0_no_aug_yolo11s_100_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 299.6±629.6 MB/s, size: 107.4 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 1.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.0it/s 23.8s
                   all       1500       7331      0.862      0.557      0.601      0.489
                banana        132        206      0.846      0.563      0.616      0.515
           bell pepper         99        369      0.876      0.596      0.663      0.559
                bulgur         22         22          1       0.35      0.401      0.367
               cabbage         11         24      0.734      0.575      0.627      0.532
                carrot   

In [ ]:
# A1 Basic Aug - 100 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A1_basic_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A1_basic_aug_yolo11s_100_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1439.6±518.1 MB/s, size: 107.4 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 1.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 3.8it/s 24.5s
                   all       1500       7331      0.793       0.37      0.502      0.379
                banana        132        206      0.927      0.368      0.585      0.439
           bell pepper         99        369      0.861      0.382      0.525      0.396
                bulgur         22         22      0.834      0.227      0.311      0.258
               cabbage         11         24      0.894      0.351      0.559      0.452
                carrot  

In [ ]:
# A2 Extended Aug - 100 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A2_extended_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A2_extended_aug_yolo11s_100_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.209 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 57.3±18.1 MB/s, size: 107.4 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels.cache... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 1.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 3.7it/s 25.1s
                   all       1500       7331      0.807      0.416      0.516      0.391
                banana        132        206      0.742      0.471      0.563      0.399
           bell pepper         99        369      0.784      0.424      0.519      0.418
                bulgur         22         22      0.777      0.227      0.281      0.236
               cabbage         11         24      0.728      0.333      0.426       0.33
                carrot     

In [14]:
# A3 Strong Aug - 100 epochs Fine-tuned YOLO Evaluation on synthetic multiple-ingredient images

from ultralytics import YOLO
import numpy as np
import os

# Define paths
YOLO_MODEL_100_PATH = os.path.join(ROOT_DIR, 'models', 'A3_strong_aug_yolo11s_100_epochs', 'weights', 'best.pt')

# Load the trained model
trained_model = YOLO(YOLO_MODEL_100_PATH)

# Evaluate the model
metrics = trained_model.val(data=MULTIPLE_INGREDIENTS_DATA_YAML_PATH, name='val_A3_strong_aug_yolo11s_100_epochs_multiple', exist_ok=True)

# Extract metrics
map50 = metrics.box.map50         # mAP@0.5
map50_95 = metrics.box.map        # mAP@0.5:0.95
precision_all = metrics.box.p     # list of precision per class
recall_all = metrics.box.r        # list of recall per class
class_names = trained_model.names # class ID to name mapping


# Compute F1-score per class
f1_scores = 2 * (precision_all * recall_all) / (precision_all + recall_all + 1e-6)  # avoid division by zero

# Compute mean F1-score
mean_f1 = np.mean(f1_scores)

# Print mean metrics
print(f"\nOverall Metrics:")
print(f"Mean Precision: {np.mean(precision_all):.4f}")
print(f"Mean Recall: {np.mean(recall_all):.4f}")
print(f"Mean F1-score: {mean_f1:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map50_95:.4f}")

# Print per-class precision and recall
print("\nPer-Class Metrics:")
for idx, (prec, rec) in enumerate(zip(precision_all, recall_all)):
    class_name = class_names[idx] if class_names and idx in class_names else f"Class {idx}"
    print(f"Class ID {idx:2d} ({class_name:<20}): Precision = {prec:.4f}, Recall = {rec:.4f}")

Ultralytics 8.3.211 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s summary (fused): 100 layers, 9,426,732 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 135.9±260.1 MB/s, size: 99.1 KB)
val: Scanning /content/multi_ingredient_dataset/valid/labels... 1500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1500/1500 417.2it/s 3.6s
val: New cache created: /content/multi_ingredient_dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.0it/s 23.7s
                   all       1500       7546      0.876      0.603      0.669      0.512
                banana        107        163      0.853      0.521      0.606      0.435
           bell pepper         84        318      0.831      0.503      0.576      0.451
                bulgur         25         25      0.991       0.84      0.846       0.72
               cabbage          4          